# Elenchus Validation Playbook

⚠️ Research software. Use at your own risk.

This notebook is a practical step-through for:
1. Comparing pre/post calibration behavior
2. Scrutinizing failure modes and reliability
3. Extending hard-to-vary testing to a coding domain

## Important Limits

- Results may be wrong, unstable, or misleading.
- This notebook is for evaluation and research only.
- Do **not** use these outputs in production decisions or production systems.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
BASE_FILE = ROOT / "benchmark_gemini_flash.json"
POST_FILE = ROOT / "benchmark_gemini_flash_postcal.json"

print('cwd:', ROOT)
print('baseline exists:', BASE_FILE.exists())
print('post-cal exists:', POST_FILE.exists())

In [ ]:
with open(BASE_FILE) as f:
    base = json.load(f)
with open(POST_FILE) as f:
    post = json.load(f)

base_s = base['summary']
post_s = post['summary']

metrics = [
    'answer_accuracy_pct',
    'answer_correct',
    'probe_errors',
    'mean_probe_score',
    'mean_mechanism_score',
    'mean_latency_s',
]

rows = []
for m in metrics:
    b = base_s.get(m)
    p = post_s.get(m)
    d = (p - b) if isinstance(b, (int, float)) and isinstance(p, (int, float)) else None
    rows.append({'metric': m, 'baseline': b, 'post_cal': p, 'delta': d})

delta_df = pd.DataFrame(rows)
delta_df

## Scrutiny Checklist

Use this in review:
- Did `probe_errors` go down?
- Did `answer_accuracy_pct` improve or regress?
- Are improvements stable across categories (not only one)?
- Did latency increase enough to matter for your evaluation budget?
- Are we reducing parser/format failures or improving reasoning quality?

Production note: even strong metrics here do **not** imply production readiness.

In [ ]:
def error_breakdown(results):
    counts = {}
    for r in results:
        err = r.get('error')
        if not err:
            continue
        key = err.split(':')[0][:80]
        counts[key] = counts.get(key, 0) + 1
    return pd.DataFrame(sorted(counts.items(), key=lambda x: x[1], reverse=True), columns=['error_prefix', 'count'])

base_err = error_breakdown(base['results'])
post_err = error_breakdown(post['results'])

print('Baseline error classes')
display(base_err.head(10))
print('Post-calibration error classes')
display(post_err.head(10))

## Add a Coding Domain (Hard-to-Vary for Code)

Idea: treat a coding task spec like a math problem.

Workflow:
1. Baseline task prompt: function + tests
2. Perturb constraints (input bounds, edge cases, performance limits)
3. Re-run generated solution against updated tests
4. Score mechanism robustness = how consistently behavior tracks changed constraints

Use this as an experiment harness, not a deployment gate.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Any

@dataclass
class CodingConstraint:
    name: str
    original: Any
    new: Any

def score_constraint_following(run_tests: Callable[[], bool], run_tests_perturbed: Callable[[], bool]):
    base_ok = run_tests()
    pert_ok = run_tests_perturbed()
    if base_ok and pert_ok:
        return 1.0
    if base_ok and not pert_ok:
        return 0.5
    return 0.0

print('Template ready: plug in your test runners and perturbed constraints.')

## Recommended Next Experiments

1. Repeat post-calibration benchmark on 3 random 20-problem slices and average
2. Track separate metrics for parser failures vs reasoning failures
3. Build one coding mini-benchmark (5 tasks, 3 perturbations each)
4. Compare calibration on Flash-only vs Flash/Pro split when budget allows
5. Add baseline delta reviews using `--compare-to` for every benchmark run